# Comércio de calçados masculinos Intent Dataset\n\nGeração e validação de dataset sintético em português do Brasil. Escala: **14.806 treino + 156 avaliação**, igual ao dataset `legal` do repositório.

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT = Path('../datasets/mens_footwear')
OUT.mkdir(parents=True, exist_ok=True)

LABELS = [
    "PRODUTO",
    "TAMANHO",
    "DISPONIBILIDADE",
    "PRECO",
    "MODELO",
    "ESTOQUE",
    "COMPRA",
    "TROCA",
    "DEVOLUCAO",
    "PAGAMENTO",
    "ENTREGA",
    "ENDERECO"
]
TRAIN_COUNTS = {label: 1234 for label in LABELS}
TRAIN_COUNTS['PRODUTO'] = 1232
EVAL_PER_LABEL = 13

PRODUCTS = [
    "um tênis masculino",
    "um sapato social masculino",
    "um mocassim masculino",
    "uma bota masculina",
    "um sapatênis masculino",
    "um chinelo masculino",
    "uma sandália masculina",
    "um coturno masculino"
]
QUALIFIERS = [
    "para homem",
    "para uso diário",
    "para trabalho",
    "para academia",
    "para sair",
    "para presente",
    "para uso social",
    "para este mês"
]

OPENERS = {
    "PRODUTO": [
        "Quero saber sobre",
        "Gostaria de informações sobre",
        "Vocês têm",
        "Queria conhecer",
        "Pode me mostrar",
        "Estou procurando",
        "Tenho interesse em",
        "Queria encontrar"
    ],
    "TAMANHO": [
        "Quero saber o tamanho de",
        "Preciso confirmar o tamanho de",
        "Qual tamanho devo escolher para",
        "Vocês têm tamanho de",
        "Queria saber a numeração de",
        "Tem como consultar o tamanho de",
        "Estou em dúvida sobre o tamanho de",
        "Pode me orientar sobre o tamanho de"
    ],
    "DISPONIBILIDADE": [
        "Tem",
        "Vocês têm",
        "Existe",
        "Consigo encontrar",
        "Dá para encontrar",
        "Queria saber se tem",
        "Gostaria de verificar se há",
        "Quero saber se está disponível"
    ],
    "PRECO": [
        "Quanto custa",
        "Qual o preço de",
        "Qual o valor de",
        "Gostaria de saber quanto custa",
        "Queria saber o preço de",
        "Pode me passar o valor de",
        "Quanto fica",
        "Tem como informar o preço de"
    ],
    "MODELO": [
        "Quero ver modelos de",
        "Gostaria de conhecer modelos de",
        "Quais modelos de",
        "Vocês trabalham com",
        "Queria opções de",
        "Estou procurando modelos de",
        "Pode me mostrar modelos de",
        "Quais estilos de"
    ],
    "ESTOQUE": [
        "Tem em estoque",
        "Está em estoque",
        "Vocês ainda têm",
        "Ainda existe estoque de",
        "Queria confirmar o estoque de",
        "Há unidades de",
        "Consegue verificar o estoque de",
        "Tem disponível no estoque"
    ],
    "COMPRA": [
        "Quero comprar",
        "Gostaria de comprar",
        "Como faço para comprar",
        "Queria comprar",
        "Posso comprar",
        "Quero pedir",
        "Tenho interesse em comprar",
        "Dá para comprar"
    ],
    "TROCA": [
        "Quero trocar",
        "Preciso trocar",
        "Como faço para trocar",
        "Gostaria de trocar",
        "Posso trocar",
        "Queria solicitar troca de",
        "Dá para trocar",
        "Preciso saber como trocar"
    ],
    "DEVOLUCAO": [
        "Quero devolver",
        "Preciso devolver",
        "Como faço para devolver",
        "Gostaria de devolver",
        "Posso devolver",
        "Queria solicitar devolução de",
        "Dá para devolver",
        "Preciso saber como devolver"
    ],
    "PAGAMENTO": [
        "Quais formas de pagamento aceitam",
        "Posso pagar",
        "Dá para pagar",
        "Como posso pagar",
        "Vocês aceitam",
        "Queria saber se aceitam",
        "É possível pagar",
        "Como funciona o pagamento de"
    ],
    "ENTREGA": [
        "Vocês entregam",
        "Qual o prazo de entrega de",
        "Quanto demora para chegar",
        "Como funciona a entrega de",
        "Queria saber sobre a entrega de",
        "Dá para entregar",
        "Vocês fazem entrega de",
        "Qual o valor da entrega de"
    ],
    "ENDERECO": [
        "Onde fica",
        "Qual o endereço de",
        "Como chego à",
        "Pode me passar o endereço de",
        "Quero saber onde fica",
        "Preciso localizar",
        "Qual a localização de",
        "Vocês ficam onde"
    ]
}

ENDINGS = ['por favor', 'para mim', 'antes de comprar', 'se possível']

def objects_for(label):
    if label == 'PAGAMENTO':
        return ['minha compra', 'o pedido', 'o produto', 'a compra', 'o pagamento', 'a encomenda', 'o pedido de ' + PRODUCTS[0], 'uma compra']
    if label == 'ENDERECO':
        return ['a loja', 'a loja física', 'o estabelecimento', 'a unidade', 'a loja de vocês', 'o ponto de venda', 'a filial', 'o comércio']
    return PRODUCTS

def generate(label, start, count):
    A, B, C, D = OPENERS[label], objects_for(label), QUALIFIERS, ENDINGS
    total = len(B) * len(C) * len(D)
    rows = []
    for k in range(start, start + count):
        ia = (k // total) % len(A)
        rem = k % total
        ib = (rem // (len(C) * len(D))) % len(B)
        ic = (rem // len(D)) % len(C)
        id_ = rem % len(D)
        text = f'{A[ia]} {B[ib]} {C[ic]}, {D[id_]}'
        text += ['.', '?', '!', '.'][k % 4]
        rows.append({'text': text, 'label': label, 'domain': 'mens_footwear'})
    return rows

train, evaluation = [], []
for label in LABELS:
    evaluation += generate(label, 0, EVAL_PER_LABEL)
    train += generate(label, EVAL_PER_LABEL, TRAIN_COUNTS[label])

(OUT / 'train.jsonl').write_text(
    '\n'.join(json.dumps(x, ensure_ascii=False) for x in train) + '\n',
    encoding='utf-8'
)
(OUT / 'eval.jsonl').write_text(
    '\n'.join(json.dumps(x, ensure_ascii=False) for x in evaluation) + '\n',
    encoding='utf-8'
)

train_df = pd.DataFrame(train)
eval_df = pd.DataFrame(evaluation)

assert len(train_df) == 14806
assert len(eval_df) == 156
assert train_df.groupby('label').size().to_dict() == TRAIN_COUNTS
assert eval_df.groupby('label').size().eq(13).all()
assert train_df['text'].is_unique
assert eval_df['text'].is_unique
assert not set(train_df['text']).intersection(eval_df['text'])
assert set(train_df['label']) == set(LABELS) == set(eval_df['label'])

print('Validation OK')
display(train_df.groupby('label').size().rename('train').to_frame())
display(train_df.sample(20, random_state=42))
display(eval_df.groupby('label').head(2))

